# 03. Flood-Frequency Uncertainty from Likelihood

Use BestFit GEV likelihood evaluations across a parameter grid to build uncertainty for Q100.

In [ ]:
import pythonnet
pythonnet.load("netfx")

import clr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helper_functions import resolve_bestfit_dll, resolve_numerics_dll, convert_to_donet_array

clr.AddReference(str(resolve_numerics_dll()))
clr.AddReference(str(resolve_bestfit_dll()))

from RMC.BestFit import DataFrame, ExactData
from RMC.BestFit.Model import UnivariateDistribution
from Numerics.Distributions import UnivariateDistributionType

In [ ]:
rng = np.random.default_rng(30)
years = np.arange(1985, 1985 + 45)
flows = np.maximum(500, 9200 + 45*(years-years.min()) + rng.normal(0, 950, len(years)))

df = DataFrame()
for y, q in zip(years, flows):
    df.ExactSeries.Add(ExactData(int(y), float(q)))

x = np.array(flows, dtype=float)
mu0, sd0 = float(np.mean(x)), float(np.std(x, ddof=1))
model = UnivariateDistribution(df, UnivariateDistributionType.GeneralizedExtremeValue)

# Parameter grid and robust likelihood evaluation
loc_grid = np.linspace(mu0 - 0.25*sd0, mu0 + 0.25*sd0, 11)
sca_grid = np.linspace(0.60*sd0, 1.00*sd0, 11)
shp_grid = np.linspace(-0.08, 0.18, 11)

def robust_loglik(m, p_arr, data):
    try:
        ll = float(m.LogLikelihood(p_arr))
        if np.isfinite(ll) and ll > -1e300:
            return ll
    except Exception:
        pass
    pdf = np.array([max(1e-300, float(m.Distribution.PDF(float(v)))) for v in data], dtype=float)
    return float(np.sum(np.log(pdf)))

params, lls = [], []
for loc in loc_grid:
    for sca in sca_grid:
        for shp in shp_grid:
            p = convert_to_donet_array([float(loc), float(max(1e-6, sca)), float(shp)])
            model.SetParameterValues(p)
            ll = robust_loglik(model, p, x)
            if np.isfinite(ll):
                params.append((loc, sca, shp))
                lls.append(ll)

lls = np.array(lls, dtype=float)
w = np.exp(lls - np.max(lls))
w = w / np.sum(w)

q100 = []
for (loc, sca, shp), wi in zip(params, w):
    model.SetParameterValues(convert_to_donet_array([loc, sca, shp]))
    q100.append((float(model.Distribution.InverseCDF(0.99)), wi))

q_vals = np.array([q for q, _ in q100], dtype=float)
q_w = np.array([ww for _, ww in q100], dtype=float)
q_mean = float(np.sum(q_vals * q_w))

print('Weighted Q100 mean:', round(q_mean, 2))
print('Q100 min/max across grid:', (round(float(np.min(q_vals)), 2), round(float(np.max(q_vals)), 2)))

In [ ]:
plt.figure(figsize=(8,4.5))
plt.hist(q_vals, bins=35, weights=q_w, alpha=0.8, color='#3d7ea6')
plt.axvline(q_mean, color='black', lw=2, label='weighted mean')
plt.title('Q100 from BestFit Likelihood Grid')
plt.xlabel('Q100')
plt.ylabel('Weighted density')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

## Summary and Exercises

You propagated BestFit parameter-likelihood structure directly into quantile uncertainty.

Exercises:
1. Increase/decrease grid resolution and compare stability.
2. Repeat for Q50 and Q500.
3. Add prior-weight multipliers to emulate informative priors.